In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, avg,col,first,when,udf,regexp_replace,to_timestamp
from pyspark.sql.types import StringType, DoubleType, ArrayType
import re
ss = SparkSession.builder.config("spark.jars", "/home/jrodarte/postgresql-42.7.3.jar").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/09/02 20:04:07 WARN Utils: Your hostname, jrodarte-ThinkPad-T480, resolves to a loopback address: 127.0.1.1; using 192.168.100.13 instead (on interface wlp3s0)
25/09/02 20:04:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/09/02 20:04:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/02 20:04:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [21]:
dfPrestamos = ss.read.format("csv").options(header='true', inferSchema='true', delimiter=',').load("documentos/mis_prestamos020925.csv")
dfPrestamos.schema


StructType([StructField('usuario_inversionista', IntegerType(), True), StructField('monto_fondeado', DoubleType(), True), StructField('plazo_meses', IntegerType(), True), StructField('tasa_anual', DoubleType(), True), StructField('calificacion', StringType(), True), StructField('estatus', StringType(), True), StructField('detalle_estatus', StringType(), True), StructField('sub_estatus', StringType(), True), StructField('ingreso_mensual', DoubleType(), True), StructField('gasto_mensual_total', DoubleType(), True), StructField('ocupacion', StringType(), True), StructField('nivel_de_estudios', StringType(), True), StructField('fecha_liberado', StringType(), True), StructField('fecha_fondeo', DateType(), True), StructField('destino', StringType(), True), StructField('id_solicitud_de_credito', IntegerType(), True), StructField('nombre_usuario_solicitante', StringType(), True), StructField('capital_pagado', DoubleType(), True), StructField('interes_ordinario_pagado', DoubleType(), True), Str

In [22]:
dfPrestamos.show()

+---------------------+--------------+-----------+----------+------------+---------+--------------------+------------+---------------+-------------------+--------------------+-----------------+--------------+------------+----------------+-----------------------+--------------------------+--------------+------------------------+------------------+------------------+----------------------+------------+-------------------+--------------------+
|usuario_inversionista|monto_fondeado|plazo_meses|tasa_anual|calificacion|  estatus|     detalle_estatus| sub_estatus|ingreso_mensual|gasto_mensual_total|           ocupacion|nivel_de_estudios|fecha_liberado|fecha_fondeo|         destino|id_solicitud_de_credito|nombre_usuario_solicitante|capital_pagado|interes_ordinario_pagado|iva_pagado_interes|moratorios_pagados|iva_moratorios_pagados|recuperacion|             estado|           municipio|
+---------------------+--------------+-----------+----------+------------+---------+--------------------+-----

In [23]:
def parceFecha(fecha):
    if fecha == '0000-00-00':
        return '1900-01-01'
    else:
        return fecha

parceFecha_udf = udf(parceFecha, StringType())

In [24]:
dfPrestamos = dfPrestamos.withColumn('fecha_liberado', parceFecha_udf(col("fecha_liberado")))
#dfPrestamos.filter(col("fecha_liberado") == '1900-01-01').show()
dfPrestamos = dfPrestamos.withColumn('fecha_liberado', to_timestamp(col("fecha_liberado"), "yyyy-MM-dd"))
dfPrestamos.show()

+---------------------+--------------+-----------+----------+------------+---------+--------------------+------------+---------------+-------------------+--------------------+-----------------+-------------------+------------+----------------+-----------------------+--------------------------+--------------+------------------------+------------------+------------------+----------------------+------------+-------------------+--------------------+
|usuario_inversionista|monto_fondeado|plazo_meses|tasa_anual|calificacion|  estatus|     detalle_estatus| sub_estatus|ingreso_mensual|gasto_mensual_total|           ocupacion|nivel_de_estudios|     fecha_liberado|fecha_fondeo|         destino|id_solicitud_de_credito|nombre_usuario_solicitante|capital_pagado|interes_ordinario_pagado|iva_pagado_interes|moratorios_pagados|iva_moratorios_pagados|recuperacion|             estado|           municipio|
+---------------------+--------------+-----------+----------+------------+---------+----------------

In [27]:
dfPrestamos.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/postgres") \
    .option("dbtable", "prestadero.prestamos") \
    .option("user", "jrodarte") \
    .option("password", "roma1993_") \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [10]:
dfDetalleMovimiento.show()

+------------+--------------------+
|autorizacion|             detalle|
+------------+--------------------+
|   104924819|                NULL|
|   104924820|                NULL|
|   104924821|Principal:4.04683...|
|   104923175|                NULL|
|   104923176|                NULL|
|   104923177|Principal:3.09670...|
|   104859707|Principal:0.00000...|
|   104732052|                NULL|
|   104732053|                NULL|
|   104732054|Principal:10.4385...|
|   104673705|                NULL|
|   104673706|                NULL|
|   104673707|Principal:0.00000...|
|   104636402|Principal:10.6304...|
|   104635868|Principal:5.31522...|
|   104634793|Principal:8.68985...|
|   104579832|Principal:12.4996...|
|   104551730|Principal:17.5966...|
|   104536750|Principal:24.0733...|
|   104530273|Principal:17.8160...|
+------------+--------------------+
only showing top 20 rows


In [11]:
ptrDigitos = r"\d\s"
ptrEspacioFinal = r"\s$"
def limpiezaDetalleMov(detalleMov):
    if detalleMov is None:
        return []
    else:
        coincidencia = re.findall(ptrDigitos, detalleMov)
        lenCoincidencia = len(coincidencia)
        arrDetalle = re.split(ptrDigitos, detalleMov)
        
        return arrDetalle

limpiezaDetalleMovUDF = udf(limpiezaDetalleMov, ArrayType(StringType()))

In [12]:
dfDetalleMovimiento = dfDetalleMovimiento.withColumn('arrDetalle', limpiezaDetalleMovUDF(dfDetalleMovimiento.detalle))
dfDetalleMovimiento.show()
#dfDetalleMovimiento.write.mode('overwrite').csv('Proyectos/prestadero/detalleMov.csv')

+------------+--------------------+--------------------+
|autorizacion|             detalle|          arrDetalle|
+------------+--------------------+--------------------+
|   104924819|                NULL|                  []|
|   104924820|                NULL|                  []|
|   104924821|Principal:4.04683...|[Principal:4.0468...|
|   104923175|                NULL|                  []|
|   104923176|                NULL|                  []|
|   104923177|Principal:3.09670...|[Principal:3.0967...|
|   104859707|Principal:0.00000...|[Principal:0.0000...|
|   104732052|                NULL|                  []|
|   104732053|                NULL|                  []|
|   104732054|Principal:10.4385...|[Principal:10.438...|
|   104673705|                NULL|                  []|
|   104673706|                NULL|                  []|
|   104673707|Principal:0.00000...|[Principal:0.0000...|
|   104636402|Principal:10.6304...|[Principal:10.630...|
|   104635868|Principal:5.31522

In [13]:
def procesarDetalleMov(aut,arrDetalle):
    arrDetalleNuevo = []
    for i in arrDetalle:
        Autitem = ("autorizacion", aut)
        Conitem = ("concepto", i.split(':')[0])
        Monitem = ("monto", i.split(':')[1])
        arrDetalleNuevo.append([Autitem,Conitem,Monitem])

    dictDetalleNuevo = dict(arrDetalleNuevo)
    return dictDetalleNuevo

In [227]:
procesarDetalleMov(103124028,["Principal: 0.0000", "Interes: 0.0000", "Impuesto Interes:  0.0000", "Moratorios: 0.0028",  "Impuesto Moratorios: 0.0004", "IVA Comisión Moratorios:  0.00000"])

ValueError: dictionary update sequence element #0 has length 3; 2 is required